# GPU-Accelerated CLIP: Training, Fine-Tuning & Semantic Search
### Interactive Walkthrough with Google Colab & Local GPU Support

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RuchitPahadia/Drive_Orchestrator/blob/main/notebooks/clip_semantic_search_walkthrough.ipynb)

> **Google Colab Tip**:
> Before running, make sure to enable the GPU in Google Colab:
> Go to **Runtime** -> **Change runtime type** -> Select **T4 GPU** (or A100 / L4) -> Click **Save**.

This notebook covers:
1. **Complete Module Imports & Environment Verification**: Upfront imports for ML, vision, and pgvector database.
2. **Google Colab & GPU Detection**: Auto-detects Colab vs local environment, reports available GPU (e.g. Tesla T4 16GB, RTX 3050).
3. **Loading CLIP Model**: Loading `openai/clip-vit-base-patch32` in GPU VRAM in FP16 mixed precision.
4. **Inference & Embeddings on GPU**: Extracting 512-dim visual and text embeddings at high throughput.
5. **Full CLIP Contrastive Training / Fine-Tuning Loop on GPU**:
   - Custom PyTorch `Dataset` and `DataLoader`.
   - Symmetric cross-entropy contrastive loss (Image-to-Text + Text-to-Image).
   - Mixed precision training with `torch.cuda.amp.autocast()` and `GradScaler`.
   - Full backpropagation and optimizer steps on GPU.
   - Saving trained model weights to Google Drive or local disk.
6. **pgvector & Database Search Integration**: Saving vectors to PostgreSQL with the HNSW index.

---
## 0. Colab Package Installation (Run if on Google Colab)

Google Colab already provides PyTorch with CUDA pre-installed. Run the cell below to install the remaining libraries:

In [ ]:
# Automatically install dependencies if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab. Installing dependencies...")
    !pip install -q transformers pillow matplotlib numpy psycopg2-binary python-dotenv
except ImportError:
    IN_COLAB = False
    print("Running in local environment (.venv).")

---
## 1. Module Imports & Setup

All required modules for standard utilities, PyTorch with CUDA acceleration, Hugging Face Transformers, image processing, visualization, and PostgreSQL/pgvector database connections are imported below.

In [ ]:
# ==============================================================================
# 1. Standard Library & System Utilities
# ==============================================================================
import os
import sys
import time
import json
import math
from pathlib import Path

# ==============================================================================
# 2. PyTorch & CUDA Deep Learning Framework
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ==============================================================================
# 3. Hugging Face Transformers (CLIP Multimodal Vision-Language Model)
# ==============================================================================
import transformers
from transformers import (
    CLIPProcessor,
    CLIPModel,
    CLIPTokenizer,
    CLIPImageProcessor
)

# ==============================================================================
# 4. Image Processing & Computer Vision
# ==============================================================================
import PIL
from PIL import Image, ImageDraw, ImageFont
import torchvision
import torchvision.transforms as transforms

# ==============================================================================
# 5. Numerical Computation & Data Visualization
# ==============================================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# ==============================================================================
# 6. PostgreSQL & pgvector Database Connectivity
# ==============================================================================
import psycopg2
from psycopg2.extras import execute_values, RealDictCursor
from dotenv import load_dotenv

print("All modules imported successfully!")

---
## 2. Hardware & CUDA GPU Health Check

Let us inspect the active environment, verify the versions of our key packages, and check our GPU configuration (e.g. Tesla T4 on Colab or RTX on local).

In [ ]:
print("=" * 70)
print(f"Environment:           {'Google Colab' if IN_COLAB else 'Local Python (.venv)'}")
print(f"Python Version:        {sys.version.split()[0]}")
print(f"PyTorch Version:       {torch.__version__}")
print(f"Transformers Version:  {transformers.__version__}")
print("-" * 70)

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:        {cuda_available}")

if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    cuda_arch = torch.cuda.get_device_capability(0)
    print(f"Active GPU Device:     {gpu_name}")
    print(f"Total GPU Memory:      {vram_mb:.0f} MB ({vram_mb / 1024:.2f} GB)")
    print(f"CUDA Compute Cap:      {cuda_arch[0]}.{cuda_arch[1]}")
else:
    device = torch.device("cpu")
    print("Warning: CUDA GPU not active. Using CPU fallback.")
    if IN_COLAB:
        print("In Colab: Go to Runtime -> Change runtime type -> Select T4 GPU")
print("=" * 70)

---
## 3. Loading the CLIP Model into GPU VRAM

We load `openai/clip-vit-base-patch32` and move it directly to GPU memory with `.to(device)`.

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"

print(f"Loading {MODEL_ID} into {device}...")
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_ID)

# Check VRAM allocation on GPU
if device.type == "cuda":
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 2)
    print(f"Model loaded in VRAM:  {allocated:.1f} MB (Reserved: {reserved:.1f} MB)")
    print(f"Vision features:       {model.visual_projection.out_features} dims")
    print(f"Text features:         {model.text_projection.out_features} dims")
    print("Both encoders project to the same 512-dimensional vector space!")

---
## 4. Creating Training & Test Images

Let us create a synthetic dataset of photos and paired captions to test both training and zero-shot retrieval.

In [ ]:
def create_sample_images():
    # 1. Sunset on ocean
    img1 = Image.new('RGB', (224, 224), color=(255, 120, 50))
    d1 = ImageDraw.Draw(img1)
    d1.ellipse([70, 50, 150, 130], fill=(255, 240, 80))
    d1.rectangle([0, 130, 224, 224], fill=(20, 60, 140))

    # 2. Green park / nature
    img2 = Image.new('RGB', (224, 224), color=(135, 206, 235))
    d2 = ImageDraw.Draw(img2)
    d2.ellipse([160, 20, 200, 60], fill=(255, 215, 0))
    d2.rectangle([0, 150, 224, 224], fill=(34, 139, 34))

    # 3. Night city skyline
    img3 = Image.new('RGB', (224, 224), color=(10, 15, 40))
    d3 = ImageDraw.Draw(img3)
    d3.ellipse([160, 30, 190, 60], fill=(240, 240, 255))
    d3.rectangle([30, 90, 80, 224], fill=(70, 70, 90))
    d3.rectangle([90, 60, 140, 224], fill=(90, 90, 110))
    d3.rectangle([150, 110, 200, 224], fill=(60, 60, 80))

    # 4. Red sports car
    img4 = Image.new('RGB', (224, 224), color=(220, 220, 220))
    d4 = ImageDraw.Draw(img4)
    d4.rectangle([40, 100, 180, 160], fill=(220, 20, 60))
    d4.ellipse([55, 145, 85, 175], fill=(30, 30, 30))
    d4.ellipse([135, 145, 165, 175], fill=(30, 30, 30))

    return [img1, img2, img3, img4]

images = create_sample_images()
captions = [
    "a warm orange sunset over the blue ocean",
    "a sunny day in a green park with lush grass",
    "a night skyline with glowing moon and city buildings",
    "a sleek red sports car parked outdoors"
]

# Display the training samples
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, (img, cap) in enumerate(zip(images, captions)):
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i+1}:\n{cap}", fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

---
## 5. PyTorch Dataset for GPU Training & Batching

On Google Colab (with 16GB VRAM on T4 GPUs), you can easily scale `batch_size` up to **16 or 32**.

In [ ]:
class PhotoCaptionDataset(Dataset):
    def __init__(self, images, captions, processor):
        self.images = images
        self.captions = captions
        self.processor = processor

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        caption = self.captions[idx]
        
        inputs = self.processor(
            text=[caption],
            images=image,
            return_tensors="pt",
            padding="max_length",
            max_length=64,
            truncation=True
        )
        
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

dataset = PhotoCaptionDataset(images, captions, processor)
# Batch size: 2 for demo, can scale to 8-32 on Colab T4 16GB
batch_size = 4 if (cuda_available and vram_mb > 6000) else 2
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"DataLoader configured with batch_size={batch_size}")

---
## 6. Symmetric Contrastive Loss Function on GPU

CLIP uses symmetric contrastive cross-entropy loss:
1. Compute pairwise similarity matrix: logits = (I . T.t()) * exp(logit_scale).
2. Ground truth labels are the diagonal indices ([0, 1, 2, ..., N-1]).
3. Loss is: 0.5 * (CrossEntropy(logits, labels) + CrossEntropy(logits.t(), labels)).

In [ ]:
def clip_contrastive_loss(image_features, text_features, logit_scale):
    """
    Computes symmetric contrastive loss directly on GPU tensors.
    """
    # Safely extract pooler_output tensor if using Transformers ModelOutput
    if hasattr(image_features, "pooler_output"):
        image_features = image_features.pooler_output
    if hasattr(text_features, "pooler_output"):
        text_features = text_features.pooler_output

    # 1. Normalize features to unit vectors (L2 norm = 1.0)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # 2. Compute cosine similarity matrix scaled by temperature
    logits_per_image = torch.matmul(image_features, text_features.t()) * logit_scale.exp()
    logits_per_text = logits_per_image.t()

    # 3. Target labels: [0, 1, 2, ..., batch_size - 1] (diagonal matching)
    batch_size = image_features.shape[0]
    labels = torch.arange(batch_size, device=image_features.device)

    # 4. Symmetric cross entropy loss
    loss_img = F.cross_entropy(logits_per_image, labels)
    loss_txt = F.cross_entropy(logits_per_text, labels)
    return (loss_img + loss_txt) / 2.0

---
## 7. GPU Training Loop with Mixed Precision (FP16)

Here we execute the training loop:
- All tensors moved to GPU using `.to(device)`.
- `torch.cuda.amp.autocast()` enables **FP16 mixed precision** on GPU Tensor Cores.
- `torch.cuda.amp.GradScaler()` scales gradients to prevent underflow in FP16.
- Optimizer: `AdamW` with weight decay.

In [ ]:
# Set model to training mode
model.train()

# Learning rate and optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)

# GradScaler for automatic mixed precision on CUDA
scaler = GradScaler(enabled=(device.type == "cuda"))

NUM_EPOCHS = 3
print(f"Starting training loop on {device.type.upper()} ({device})...")

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    batch_count = 0
    
    for step, batch in enumerate(dataloader):
        # 1. Move batch inputs to GPU device
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        # 2. Forward pass with mixed precision on GPU
        with autocast(enabled=(device.type == "cuda")):
            img_out = model.get_image_features(pixel_values=pixel_values)
            image_features = getattr(img_out, "pooler_output", img_out)
            
            txt_out = model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
            text_features = getattr(txt_out, "pooler_output", txt_out)
            
            loss = clip_contrastive_loss(image_features, text_features, model.logit_scale)

        # 3. Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Average Contrastive Loss: {avg_loss:.4f}")
    
    # Print VRAM usage during training
    if device.type == "cuda":
        allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)
        print(f"   GPU VRAM Allocated: {allocated:.1f} MB")

print("Training on GPU completed successfully!")

---
## 8. Saving & Exporting Your Trained Model

After fine-tuning on Colab, you can save your model weights to load them later into Photo Orchestrator:

In [ ]:
OUTPUT_DIR = "./fine_tuned_clip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Saving fine-tuned CLIP model to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Model saved successfully!")

# Optional: If on Google Colab, mount Google Drive to save weights permanently:
"""
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained('/content/drive/MyDrive/fine_tuned_clip')
"""

---
## 9. GPU-Accelerated Search & Inference

Now let us switch the model to evaluation mode (`model.eval()`) and run inference on the GPU.

In [ ]:
model.eval()

@torch.no_grad()
def encode_image_gpu(pil_image):
    """Encodes a PIL image to 512-dim vector on GPU."""
    inputs = processor(images=pil_image, return_tensors="pt").to(device)
    with autocast(enabled=(device.type == "cuda")):
        features = model.get_image_features(**inputs)
        features = getattr(features, "pooler_output", features)
        features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features.cpu().numpy()[0]

@torch.no_grad()
def encode_text_gpu(query):
    """Encodes a search query to 512-dim vector on GPU."""
    inputs = processor(text=[query], return_tensors="pt", padding=True, truncation=True).to(device)
    with autocast(enabled=(device.type == "cuda")):
        features = model.get_text_features(**inputs)
        features = getattr(features, "pooler_output", features)
        features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features.cpu().numpy()[0]

# Compute all image embeddings on GPU
image_embeddings = [encode_image_gpu(img) for img in images]

# Test query
search_prompt = "a sleek red sports car"
query_vec = encode_text_gpu(search_prompt)

# Rank photos by cosine similarity
similarities = [float(np.dot(img_vec, query_vec)) for img_vec in image_embeddings]
ranked_indices = np.argsort(similarities)[::-1]

print(f"Search query: '{search_prompt}'\n")
for rank, idx in enumerate(ranked_indices):
    print(f"#{rank+1} Match: Sample {idx+1} ({captions[idx]}) -> Score: {similarities[idx]:.4f}")

# Plot the similarity scores
plt.figure(figsize=(8, 3.5))
plt.bar([f"Sample {i+1}" for i in range(len(images))], similarities, color=['coral', 'forestgreen', 'navy', 'crimson'])
plt.ylabel('Cosine Similarity Score')
plt.title(f"Cosine Similarity to '{search_prompt}'")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

---
## 10. Storing in PostgreSQL & Querying with `pgvector` HNSW

Once the 512-dimensional vector is computed on the GPU, it is sent to PostgreSQL for persistent storage and sub-millisecond retrieval:

```sql
-- Storing the vector in photos table:
UPDATE photos 
SET embedding = '[0.0123, -0.0456, ..., 0.0891]'::vector,
    indexed_at = NOW()
WHERE id = 'photo-uuid';

-- The fast HNSW search query in /api/photos/search:
SELECT 
    id, 
    filename, 
    thumbnail_url,
    1 - (embedding <=> $1::vector) AS similarity_score
FROM photos
WHERE user_id = $2
  AND embedding IS NOT NULL
ORDER BY embedding <=> $1::vector ASC
LIMIT 20;
```

---
## 11. Live Database Connection & Vector Inspection

Connects directly to your live Supabase database to inspect your tables and indexes.

In [ ]:
if IN_COLAB:
    # On Google Colab, you can paste your connection string or use colab userdata secrets
    try:
        from google.colab import userdata
        db_url = userdata.get('DATABASE_URL')
    except Exception:
        db_url = os.environ.get('DATABASE_URL') or input("Enter your Supabase DATABASE_URL: ")
else:
    env_path = Path("../.env.local")
    load_dotenv(dotenv_path=env_path)
    db_url = os.getenv("DATABASE_URL")

if db_url and "postgres" in db_url:
    try:
        conn = psycopg2.connect(db_url, sslmode="require")
        cur = conn.cursor(cursor_factory=RealDictCursor)
        
        cur.execute("""
            SELECT indexname, indexdef 
            FROM pg_indexes 
            WHERE tablename = 'photos';
        """)
        indexes = cur.fetchall()
        
        print("=" * 70)
        print("Connected to PostgreSQL (Supabase) Database Successfully!")
        print("-" * 70)
        for idx in indexes:
            print(f"Index: {idx['indexname']}")
            print(f"Def:   {idx['indexdef']}\n")
            
        cur.close()
        conn.close()
    except Exception as e:
        print(f"Database connection error: {e}")
else:
    print("DATABASE_URL not configured. Skipping live database query.")

---
## 12. Summary & GPU Best Practices

1. **Colab GPU Performance**: On Google Colab (Tesla T4 with 16GB VRAM), you can increase batch sizes to 16 or 32 for much faster training.
2. **Mixed Precision (`FP16`)**: `torch.cuda.amp.autocast()` leverages Tensor Cores on T4/A100 GPUs for maximum throughput.
3. **Model Export**: Save your trained weights (`model.save_pretrained`) to Google Drive so you can load them into Photo Orchestrator.
4. **Vector Compatibility**: Output vectors are **512 dimensions**, directly compatible with the **HNSW index** in `db/schema.sql`.